In [4]:
!D:/AWS_CLI/aws.exe sso login --profile rmit

Attempting to open your default browser.
If the browser does not open, open the following URL:

https://oidc.ap-southeast-2.amazonaws.com/authorize?response_type=code&client_id=Q2DTXe-JRCw1BDUzRL6NBGFwLXNvdXRoZWFzdC0y&redirect_uri=http%3A%2F%2F127.0.0.1%3A61989%2Foauth%2Fcallback&state=c116ea9f-84ac-46a2-a15a-fcab39d9a445&code_challenge_method=S256&scopes=sso%3Aaccount%3Aaccess&code_challenge=A6lsFqYmVWXzK5YL7IPihc_c8KjcNMIOgpSISANWSTs
Successfully logged into Start URL: https://rmit-research.awsapps.com/start/#


In [11]:
import asyncio
import importlib.util
import os
from pathlib import Path

In [12]:
# Jupyter sets the current working directory to the folder containing the notebook.
# However, the original python script expects to run from the project root.
# We also make sure this behaves safely if the cell is run multiple times.
cwd = Path.cwd().resolve()
if cwd.name == "label" and cwd.parent.name == "scripts":
    os.chdir(cwd.parent.parent)
    print(f"Changed working directory to project root: {Path.cwd()}")
else:
    # Provide a fallback just in case it is already run from the project root
    print(f"Current working directory is already: {Path.cwd()}")

Current working directory is already: D:\Work\Research_Project\anaconda_research_project


In [ ]:
# =========================================================
# Config
# =========================================================

# Existing script path (relative to the project root)
TARGET_SCRIPT = "trec_label_concurrent_defended.py"

# Model settings
MAX_TOKENS = 2000
TARGET_PATH = Path("scripts") / "label" / TARGET_SCRIPT

# Languages only
LANGUAGES = [
    #"ru_instruct",
    #"zh_instruct",
    "ga_instruct",
    "ar_instruct",
    "fr_instruct",
    "vi_instruct",
    "sw_instruct",
    "ga_instruct",
    "eng_instruct",
    "hi_instruct",
    "he_instruct",
]

# Shared part range for all languages
START_PART = 1
END_PART = 6

In [14]:
# =========================================================
# Load target script dynamically
# =========================================================

if not TARGET_PATH.exists():
    raise FileNotFoundError(f"Could not find target script: {TARGET_PATH.resolve()}")

spec = importlib.util.spec_from_file_location("label_runner_target", TARGET_PATH)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)


def set_run_globals(mod, lang: str, start_part: int, end_part: int) -> None:
    mod.LANG = lang
    mod.START_PART = start_part
    mod.END_PART = end_part
    if hasattr(mod, "INFERENCE_CONFIG"):
        mod.INFERENCE_CONFIG["maxTokens"] = MAX_TOKENS
    else:
        mod.INFERENCE_CONFIG = {"maxTokens": MAX_TOKENS, "temperature": 0.0, "topP": 1.0}

    if lang == "raw":
        mod.PART_DIR = Path(f"retrieved/trec_dl_{mod.TREC_DL_YEAR}/judged/")
    else:
        mod.PART_DIR = Path(f"retrieved/trec_dl_{mod.TREC_DL_YEAR}/{lang}/")

    mod.PART_PATTERN = f"all_topics_trecdl_{mod.TREC_DL_YEAR}_part{{n}}.csv"

In [ ]:
async def run_all_languages():
    for lang in LANGUAGES:
        print("\n" + "=" * 80)
        print(
            f"[RUNNER] Starting language={lang} | "
            f"parts={START_PART}..{END_PART}"
        )
        print("=" * 80)

        set_run_globals(module, lang, START_PART, END_PART)

        try:
            await module.main()
            print(f"[RUNNER] Finished language={lang}")
        except KeyboardInterrupt:
            print(f"\n[RUNNER] Interrupted while processing language={lang}")
            break
        except Exception as e:
            print(f"[RUNNER] Error while processing language={lang}: {e}")

# Run the main function. Notice we await directly since IPython already runs an event loop.
await run_all_languages()


[RUNNER] Starting language=ru_instruct | parts=1..6
[STOP] Press 'Q' to stop gracefully.

--- Running inference for model: openai.gpt-oss-20b-1:0 (run_id=20260326_162003, LANG=ru_instruct, DEFEND_LANG=judged, mode=replace) ---
[STOP] Press 'Q' at any time to stop after the current in-flight items.
[all_topics_trecdl_2022_part6.csv] Loaded ENG lookup: all_topics_trecdl_2022_part6.csv (155 rows)
[all_topics_trecdl_2022_part1.csv] Loaded ENG lookup: all_topics_trecdl_2022_part1.csv (500 rows)
[all_topics_trecdl_2022_part5.csv] Loaded ENG lookup: all_topics_trecdl_2022_part5.csv (500 rows)
[all_topics_trecdl_2022_part2.csv] Loaded ENG lookup: all_topics_trecdl_2022_part2.csv (500 rows)
[all_topics_trecdl_2022_part4.csv] Loaded ENG lookup: all_topics_trecdl_2022_part4.csv (500 rows)
[all_topics_trecdl_2022_part3.csv] Loaded ENG lookup: all_topics_trecdl_2022_part3.csv (500 rows)
[all_topics_trecdl_2022_part4.csv] Loaded 512 rows
[HEADER] LANG='ru_instruct' | output columns = ['qid', 'query

CancelledError: 

[ERROR] all_topics_trecdl_2022_part2.csv: API failed on row 148, pid_resolved=msmarco_passage_28_445177142 :: Error when retrieving token from sso: Token has expired and refresh failed[all_topics_trecdl_2022_part4.csv] [89/512] tokens in/out += 0/0 (totals 0/0)
[ERROR] all_topics_trecdl_2022_part3.csv: API failed on row 147, pid_resolved=msmarco_passage_00_826730292 :: Error when retrieving token from sso: Token has expired and refresh failed
[ERROR] all_topics_trecdl_2022_part1.csv: API failed on row 163, pid_resolved=msmarco_passage_47_257937911 :: Error when retrieving token from sso: Token has expired and refresh failed
[ERROR] all_topics_trecdl_2022_part4.csv: API failed on row 90, pid_resolved=msmarco_passage_64_774183061 :: Error when retrieving token from sso: Token has expired and refresh failed
[ERROR] all_topics_trecdl_2022_part5.csv: API failed on row 160, pid_resolved=msmarco_passage_04_239106261 :: Error when retrieving token from sso: Token has expired and refresh failed

[ERROR] all_topics_trecdl_2022_part2.csv: API failed on row 149, pid_resolved=msmarco_passage_59_598080749 :: Error when retrieving token from sso: Token has expired and refresh failed
[ERROR] all_topics_trecdl_2022_part3.csv: API failed on row 148, pid_resolved=msmarco_passage_29_466399476 :: Error when retrieving token from sso: Token has expired and refresh failed
[ERROR] all_topics_trecdl_2022_part1.csv: API failed on row 164, pid_resolved=msmarco_passage_47_259259184 :: Error when retrieving token from sso: Token has expired and refresh failed
[ERROR] all_topics_trecdl_2022_part6.csv: API failed on row 89, pid_resolved=msmarco_passage_03_344453924 :: Error when retrieving token from sso: Token has expired and refresh failed
[ERROR] all_topics_trecdl_2022_part4.csv: API failed on row 91, pid_resolved=msmarco_passage_53_711939151 :: Error when retrieving token from sso: Token has expired and refresh failed
[ERROR] all_topics_trecdl_2022_part5.csv: API failed on row 161, pid_resolved